# Chapter 6 · Monte Carlo

*Book 3 · Probability · Shelf I*

**Builds on:** [Chapter 5](05-law-of-large-numbers-and-clt.ipynb) -- the error of an average.
**You will:** compute a number by random sampling, measure its error, reach a rare probability
cheaply with importance sampling, and validate a model of the slope by simulating it two ways.

**Monte Carlo** estimates an expectation by averaging random draws. Its error falls as
$1/\sqrt N$ regardless of the dimension, which is why it wins wherever a formula is hard and a
simulation is easy.

In [1]:
import numpy as np
from scipy import stats

from ds_toolbox import rockfall as rf
from ds_toolbox import viz

viz.use_style()
rng = np.random.default_rng(6)

## Pi from random points

The share of random points in the unit square that fall inside the quarter circle is $\pi/4$.

In [2]:
for n in (1_000, 100_000, 10_000_000):
    pts = rng.random((n, 2))
    est = 4 * ((pts ** 2).sum(axis=1) <= 1).mean()
    expected = 4 * np.sqrt(np.pi / 4 * (1 - np.pi / 4) / n)
    print(f"N = {n:>10,}: pi ~ {est:.5f}  (error {abs(est - np.pi):.5f}, expected ~ {expected:.5f})")

N =      1,000: pi ~ 3.14800  (error 0.00641, expected ~ 0.05193)
N =    100,000: pi ~ 3.13924  (error 0.00235, expected ~ 0.00519)
N = 10,000,000: pi ~ 3.14233  (error 0.00074, expected ~ 0.00052)


## A rare probability: the energetic rock

Zone 2 of the slope: exponential masses (mean 115 kg) and normal speeds (37 m/s, sd 5). How likely
is a single rock at 1000 kJ or more? The toolbox has the exact value from one integral; naive Monte
Carlo sees the event only a few times.

In [3]:
zone2 = rf.true_zones()[1]
exact = rf.p_energy_at_least(zone2.mass, zone2.speed, rf.BREAK_KJ)
n = 200_000
m = zone2.mass.rvs(size=n, random_state=rng)
v = zone2.speed.rvs(size=n, random_state=rng)
hits = rf.energy_kj(m, v) >= rf.BREAK_KJ
print(f"exact P(E >= 1000 kJ) = {exact:.3e}")
print(f"naive Monte Carlo, {n:,} rocks: {hits.mean():.3e} from {hits.sum()} hits "
      f"(relative error ~ {1 / np.sqrt(max(hits.sum(), 1)):.0%})")

exact P(E >= 1000 kJ) = 4.222e-05
naive Monte Carlo, 200,000 rocks: 1.500e-05 from 3 hits (relative error ~ 58%)


## Importance sampling

Draw the masses from a distribution that makes heavy rocks *common* -- an exponential with a much
larger mean -- and weight each draw by how much likelier it is under the real distribution than
under the one we drew from: $w = f(m)/g(m)$. The weighted average is still unbiased, and now most
draws say something about the tail.

In [4]:
proposal = stats.expon(scale=800)
m_is = proposal.rvs(size=n, random_state=rng)
v_is = zone2.speed.rvs(size=n, random_state=rng)
w = zone2.mass.pdf(m_is) / proposal.pdf(m_is)
h = (rf.energy_kj(m_is, v_is) >= rf.BREAK_KJ) * w
est, se = h.mean(), h.std() / np.sqrt(n)
print(f"importance sampling, {n:,} draws: {est:.3e} +/- {se:.1e} (relative error {se / est:.1%})")

importance sampling, 200,000 draws: 4.202e-05 +/- 1.1e-06 (relative error 2.6%)


Same number of draws; the relative error falls from more than half to a few percent. Choosing the
proposal is the art: it must cover the region that matters and not explode the weights elsewhere.

## Validating a model two ways

The challenge needs the slope's breach rate: rocks through the net per year. `rockfall` computes it
two ways -- (a) by simulating every rock and the net's load day by day, (b) from integrals for the
energetic rocks plus a simulated share for the 500-1000 kJ rocks that meet a full net. If the two
disagree, one of them is wrong.

In [5]:
zones = rf.true_zones()
years = 100_000
ev = rf.simulate(zones, rf.MIN_PER_YEAR * years, seed=12)
direct = rf.through_net(ev).sum() / years
direct_se = np.sqrt(rf.through_net(ev).sum()) / years
hybrid = rf.estimate_breach_rate(zones, years=20_000, seed=13)
print(f"(a) every rock simulated, {years:,} years: {direct:.4f} +/- {direct_se:.4f} per year")
print(f"(b) integrals + the full-net share:          {hybrid:.4f} per year")
print(f"of which rocks of 1000 kJ or more: {rf.breach_rate_analytic(zones, 0):.4f} per year")

(a) every rock simulated, 100,000 years: 0.0165 +/- 0.0004 per year
(b) integrals + the full-net share:          0.0164 per year
of which rocks of 1000 kJ or more: 0.0053 per year


The two methods agree within the simulation's error, so either can stand in for the other -- and
the cheaper hybrid is what the challenge runs hundreds of times. The split is the surprise: only a
third of the breaches are rocks of 1000 kJ or more. Two thirds are 500-1000 kJ rocks that arrive
while the net is full. A model that kept only the energy threshold, and dropped the report's
second rule, would understate the risk threefold.

## What to remember

- Monte Carlo averages random draws; its error is $\sigma/\sqrt N$ in any dimension.
- Rare events starve naive sampling. Importance sampling moves the draws to where the event is.
- Validate a simulation against a second method; agreement is evidence, disagreement is a bug.

## Exercises

1. Use a proposal exponential with mean 300 and with mean 3000. Which is better, and why?
2. Estimate $\pi$ to six digits. How many points does the $1/\sqrt N$ rule say you need?
3. Change the net's clearing from daily to every two days. How does the breach rate change?

**Next:** [The challenge · rockfall](challenge/rockfall.ipynb)